In [ ]:
!pip install pandas
!pip install -U langchain-chroma
!pip install -U langchain_huggingface
!pip install -U sentence-transformers langchain.tools
!pip install langgraph
!pip install grandalf
!pip install langchain_graph groq langchain_groq
!pip install langgraph langchain_groq langchain_chroma langchain_huggingface pydantic typing-extensions

ERROR: Could not find a version that satisfies the requirement langchain_graph (from versions: none)
ERROR: No matching distribution found for langchain_graph


In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from pydantic import BaseModel, Field
from typing import Annotated
from typing_extensions import TypedDict

class State(TypedDict):
    messages: Annotated[list[str], add_messages]
    user_input: str
    message_type: str
    ticket_number: int


from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0, api_key="<api_key>")

c:\repos\AgileFever\langgraph_capstone\.venv\lib\site-packages\langgraph\cache\base\__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
c:\repos\AgileFever\langgraph_capstone\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from typing import Literal


class MessageClassifier(BaseModel):
    message_type: Literal["generic_query_resolver", "positive_feedback_resolver", "negative_feedback_resolver", "new_ticket_generator"] = Field(description='message type')


In [ ]:
import os
import pandas as pd

def positive_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Positive Feedback resolver invoked')

    ticket_number = state.get('ticket_number')

    system_prompt = f'\
                    Respond to the positive feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    and indicate that ticket_number {ticket_number} has been marked as resolved'

    df = pd.read_csv('test_data.csv')

    df.loc[df['ticket_number']==state['ticket_number'][0], 'status']='Resolved'
    df.to_csv('test_data.csv', index=False)

    positive_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(positive_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


In [ ]:
import os
import pandas as pd

def new_ticket_generator(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('New ticket resolver invoked')

    df = pd.read_csv('test_data.csv')
    new_ticket_number = df['ticket_number'].astype(int).iloc[-1] + 1


    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} to create a new summary for the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content

    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": user_feedback
        }
    ]

    summary = llm.invoke(summarizer_prompt)

    system_prompt = f'\
                    Respond to the feedback received by the user. \
                    Greet them with a response indicating the pleasure of interacting with them and conducting business with them\
                    Analyze the feedback, if it is positive, provide a friendly greeting expressing gratitude for the feedback\
                    if feedback is negative\
                    and indicate that ticket_number:{new_ticket_number} has been created with this summary: {summary}\
                    if feedback cannot be classified, give a friendly message stating you are not able to process the query and ask how you can help'
    df.loc[len(df)] = [new_ticket_number, 'Unresolved', summary.content.replace("\n", "")]

    df.to_csv('test_data.csv', index=False)

    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":new_ticket_number}


In [ ]:
from collections import abc
import os
import pandas as pd

def generic_query_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly current status of ticket raised by user and express gratitude for their correspondence
    '''
    print('Generic Query resolver invoked')
    df = pd.read_csv('test_data.csv')
    ticket_number = state.get('ticket_number')
    current_row = df[df['ticket_number']==ticket_number]
    current_status = current_row['status']
    notes = current_row['notes']


    system_prompt = f'Greet with a friendly message acknowledging the user input For the ticket_number: {ticket_number}, current_status: {current_status} and notes: {notes} \
                    Generate a summary and provide a response\
                    if any of the values are missing simply provide a friendly message indicating no recordws for the ticket_number: {ticket_number} could be found'


    generic_query_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": state['messages'][0].content
        }
    ]

    result = llm.invoke(generic_query_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":ticket_number}


In [ ]:
import os
import pandas as pd

def negative_feedback_resolver(state: State) -> str:
    '''
    Use this tool to provide a friendly response expressing gratitude towards the user upon successful resolution of their ticket
    '''
    print('Negative Feedback resolver invoked')

    # We will need 2 llm calls here
    # 1. Summarize and update the existing status
    # 2. Provide a response to the user

    summarizer_system_prompt = """
                    Summarize {user_feedback} and {current_notes} to create a new summary for the status of the ticket
                    The summary should be less than 50 words
                    """

    df = pd.read_csv('test_data.csv')
    user_feedback=state['messages'][0].content
    ticket_number = state.get('ticket_number')
    current_notes = df[df['ticket_number']==ticket_number]['notes'].item()
    summarizer_prompt = [
        {
            "role": "system",
            "content": summarizer_system_prompt
        },
        {
            "role": "user",
            "content": f'user_feedback: {user_feedback} current_notes: {current_notes}'
        }
    ]

    summarized_notes = llm.invoke(summarizer_prompt).content.replace("\n", "")
    df.loc[df['ticket_number']==ticket_number, 'status']='Unresolved'
    df.loc[df['ticket_number']==ticket_number, 'notes']=summarized_notes
    df.to_csv('test_data.csv', index=False)
    system_prompt = f'\
                    Analyze the negative feedback provider by customer in {user_feedback}, provide the summary you captured in the {summarized_notes}\
                    and provide a apologetic response indicating you are working earnestly to address their issue\
                    '
    negative_feedback_prompt = [
        {
            "role": "system",
            "content": system_prompt
        }
    ]

    result = llm.invoke(negative_feedback_prompt)
    return {"messages": [{"content": result.content, "role": "assistant"}], "ticket_number":state.get("ticket_number")}


In [ ]:
def router_message(state: State):
    user_input = state['messages'][0].content
    classifier_llm = llm.with_structured_output(MessageClassifier)
    ticket_number = state.get("ticket_number")
    system_prompt = """
                    Classify user message as 'positive_feedback_resolver', 'negative_feedback_resolver' or 'generic_query_resolver' based on the criteria below
                    'positive_feedback_resolver': if the user has provided input that indicates they are satisfied or happy with the resolution of their ticket, eg:  “Thanks for resolving my issue!"
                    'negative_feedback_resolver': if the user is still dissatisfied by the resolution Eg: “My problem still isn't fixed”
                    'generic_query_resolver': if user needs generic information regarding existing tickets or if they are seeking resolution for a brand new issue
                    'new_ticket_generator': if there is {ticket_number} is undefined/null and user is providing a brand new ticket
                    """

    final_prompt = [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_input
        }
    ]


    result = classifier_llm.invoke(final_prompt)
    return {"messages": [{"content": result.message_type, "role":"assistant"}], "user_input":user_input, "message_type":result.message_type, "ticket_number":ticket_number}

In [ ]:
def final_response(state: State):
    user_input = state["user_input"]
    context = state["messages"][-1].content # the last message is the final response
    ticket_number = state["ticket_number"]

    final_prompt = f'''
                    refer to the query and context below to build the response
                    if context is not aligned with the user query, the response must be 'No Context found'. Do not use
                    any additional context
                    user_query : {user_input}
                    ticket_number: {ticket_number}
                    context: {context}
                    '''
    result = llm.invoke(final_prompt)

    return {"messages": [{"content":result.content, "role":"assistant"}]}


In [12]:
graph_builder = StateGraph(State)
graph_builder.add_node("router", router_message)
graph_builder.add_node("positive_feedback_resolver", positive_feedback_resolver)
graph_builder.add_node("negative_feedback_resolver", negative_feedback_resolver)
graph_builder.add_node("generic_query_resolver", generic_query_resolver)
graph_builder.add_node("new_ticket_generator", new_ticket_generator)

graph_builder.add_node("final_response", final_response)

graph_builder.add_edge(START, "router")
graph_builder.add_conditional_edges("router", lambda state: state.get("message_type"), {"positive_feedback_resolver": "positive_feedback_resolver", "negative_feedback_resolver": "negative_feedback_resolver", "generic_query_resolver": "generic_query_resolver", "new_ticket_generator": "new_ticket_generator"})
graph_builder.add_edge("positive_feedback_resolver", "final_response")
graph_builder.add_edge("negative_feedback_resolver", "final_response")
graph_builder.add_edge("new_ticket_generator", "final_response")
graph_builder.add_edge("generic_query_resolver", "final_response")

graph = graph_builder.compile()

graph.get_graph().print_ascii()


                                                              +-----------+                                                                  
                                                              | __start__ |                                                                  
                                                              +-----------+                                                                  
                                                                     *                                                                       
                                                                     *                                                                       
                                                                     *                                                                       
                                                                +--------+                                                                   
      

In [13]:
result2 = graph.invoke({'messages':[{"role": "user", "content":"I am extremely displeased with your service"}], 'ticket_number':1064})
result2['messages']

Negative Feedback resolver invoked


[HumanMessage(content='I am extremely displeased with your service', additional_kwargs={}, response_metadata={}, id='3f01f25f-7ee7-4d84-b3bb-69632e5cda38'),
 AIMessage(content='negative_feedback_resolver', additional_kwargs={}, response_metadata={}, id='48fe0894-4044-4e94-b2f7-6e6e7d5e977a', tool_calls=[], invalid_tool_calls=[]),
 AIMessage(content='**Summary of the feedback**  \n- **Customer sentiment:** Extremely displeased with the service.  \n- **Current ticket status:** Pending retailer tax‑ID verification to confirm tax‑exempt status for a bulk purchase.  \n\n---\n\n**Apologetic response**\n\nDear [Customer Name],\n\nI’m truly sorry to hear about your disappointment with our service. I understand how frustrating it can be when a purchase you’re planning doesn’t move forward as expected.\n\nPlease know that we are actively working on your ticket. Our team is currently verifying the retailer tax‑ID you provided so we can confirm your tax‑exempt status for the bulk order. This step 

In [14]:
result3 = graph.invoke({'messages':[{"role": "user", "content":"what is the current status of this ticket"}], 'ticket_number':1006})
result3

Generic Query resolver invoked


{'messages': [HumanMessage(content='what is the current status of this ticket', additional_kwargs={}, response_metadata={}, id='2066418e-ad75-4483-8442-a0cca2542d4d'),
  AIMessage(content='generic_query_resolver', additional_kwargs={}, response_metadata={}, id='d8acb0e2-81b6-41e2-8793-54dff83f70c4', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='**Ticket Summary (ID:\u202f1006)**  \n\n- **Current Status:**\u202f`in_progress` (status code\u202f5)  \n- **Notes:**\u202fBuilding a custom segment for retailers who …  \n\n**Response:**  \nThe ticket you asked about (\u202f#1006\u202f) is currently **in progress**. The latest note indicates that work is underway to build a custom segment for retailers. If you need more details or have any other questions, just let me know!', additional_kwargs={}, response_metadata={}, id='3784de29-deee-4e50-b3c9-e1e7ef4aca5f', tool_calls=[], invalid_tool_calls=[]),
  AIMessage(content='The ticket you asked about (\u202f#1006\u202f) is currently *

In [17]:
result4 = graph.invoke({'messages':[{"role": "user", "content":"My diecast car was never delivered"}]})
result4['messages'][-1]

New ticket resolver invoked


AIMessage(content='I’m sorry to hear that your diecast car has not arrived. I’ve created **ticket\u202f#1102** for you with the following summary:\n\n**Summary:** *Customer reports non‑delivery of purchased diecast car.*\n\nWe’ll investigate this right away and keep you updated. If there’s anything else you need, just let me know!', additional_kwargs={}, response_metadata={}, id='1e3cd8e8-0134-48b8-ad2f-0f4be74eede0', tool_calls=[], invalid_tool_calls=[])